In [1]:
import time
import secrets
import json
from datetime import datetime

class ZeroTrustGate:
    def __init__(self):
        self.user_profiles = {
            "renad_admin": {"trusted_country": "SA", "role": "Admin"},
            "ahmed_dev": {"trusted_country": "SA", "role": "Developer"}
        }
        self.active_otps = {}

    def _emit_telemetry(self, event_type: str, status: str, details: dict):
        """توليد سجل أمني بصيغة JSON مطابق لأنظمة SIEM و Azure Sentinel / Okta"""
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "event_type": event_type,
            "status": status,
            "system_component": "ZeroTrust_Adaptive_Gate",
            "details": details
        }
        # طباعة السجل بصيغة JSON آلياً
        print(json.dumps(log_entry, ensure_ascii=False))

    def generate_otp(self, username: str) -> str:
        otp = f"{secrets.randbelow(900000) + 100000}"
        self.active_otps[username] = {
            "code": otp,
            "expiry": time.time() + 30
        }
        self._emit_telemetry("MFA_DISPATCH", "SUCCESS", {
            "user": username,
            "delivery_channel": "SMS_TRUSTED_DEVICE"
        })
        return otp

    def evaluate_access_request(self, username: str, country: str, request_time: int) -> bool:
        if username not in self.user_profiles:
            self._emit_telemetry("IDENTITY_VALIDATION", "FAILURE", {
                "user": username,
                "reason": "UserPrincipalNotFound"
            })
            return False

        user = self.user_profiles[username]
        risk_score = 0
        risk_factors = []

        if country != user["trusted_country"]:
            risk_score += 50
            risk_factors.append(f"GeographicAnomaly({country})")

        if 0 <= request_time <= 5:
            risk_score += 30
            risk_factors.append(f"UnusualTimeWindow({request_time}:00)")

        # اتخاذ القرار وفق سياسات Conditional Access
        if risk_score >= 80:
            self._emit_telemetry("CONDITIONAL_ACCESS_EVALUATION", "BLOCKED", {
                "user": username,
                "risk_score": risk_score,
                "risk_factors": risk_factors,
                "enforced_policy": "DenyAccessHighRisk"
            })
            return False

        elif risk_score >= 30:
            self._emit_telemetry("CONDITIONAL_ACCESS_EVALUATION", "CHALLENGE_REQUIRED", {
                "user": username,
                "risk_score": risk_score,
                "risk_factors": risk_factors,
                "enforced_policy": "RequireMFA_MediumRisk"
            })
            otp = self.generate_otp(username)
            # محاكاة لإدخال الرمز الصحيح
            return self.verify_mfa(username, otp)

        else:
            self._emit_telemetry("CONDITIONAL_ACCESS_EVALUATION", "ALLOWED", {
                "user": username,
                "risk_score": risk_score,
                "enforced_policy": "GrantAccessLowRisk"
            })
            return True

    def verify_mfa(self, username: str, input_code: str) -> bool:
        if username not in self.active_otps:
            return False
        
        otp_data = self.active_otps[username]
        if secrets.compare_digest(otp_data["code"], input_code):
            self._emit_telemetry("MFA_VERIFICATION", "SUCCESS", {"user": username})
            del self.active_otps[username]
            return True
        return False

if __name__ == "__main__":
    gate = ZeroTrustGate()
    print("--- [SIMULATING AZURE AD / OKTA CONDITIONAL ACCESS LOGS] ---")
    gate.evaluate_access_request("renad_admin", country="SA", request_time=10)
    gate.evaluate_access_request("ahmed_dev", country="SA", request_time=2)
    gate.evaluate_access_request("renad_admin", country="US", request_time=1)

--- [SIMULATING AZURE AD / OKTA CONDITIONAL ACCESS LOGS] ---
{"timestamp": "2026-10-03T06:58:43.040769", "event_type": "CONDITIONAL_ACCESS_EVALUATION", "status": "ALLOWED", "system_component": "ZeroTrust_Adaptive_Gate", "details": {"user": "renad_admin", "risk_score": 0, "enforced_policy": "GrantAccessLowRisk"}}
{"timestamp": "2026-10-03T06:58:43.040769", "event_type": "CONDITIONAL_ACCESS_EVALUATION", "status": "CHALLENGE_REQUIRED", "system_component": "ZeroTrust_Adaptive_Gate", "details": {"user": "ahmed_dev", "risk_score": 30, "risk_factors": ["UnusualTimeWindow(2:00)"], "enforced_policy": "RequireMFA_MediumRisk"}}
{"timestamp": "2026-10-03T06:58:43.040769", "event_type": "MFA_DISPATCH", "status": "SUCCESS", "system_component": "ZeroTrust_Adaptive_Gate", "details": {"user": "ahmed_dev", "delivery_channel": "SMS_TRUSTED_DEVICE"}}
{"timestamp": "2026-10-03T06:58:43.040769", "event_type": "MFA_VERIFICATION", "status": "SUCCESS", "system_component": "ZeroTrust_Adaptive_Gate", "details": 